
# Data Mining & Machine Learning — Teaching Week 4 Seminar
## Classification I: Probabilities, Thresholds and Business Decisions

**Anchor dataset:** IBM Telco Customer Churn  
**Main algorithms:** Logistic Regression and K-Nearest Neighbours  
**Pipeline focus:** Business → Data → Preprocessing → Baseline → Model → Validation → Decision → Test

### Seminar purpose

This seminar is deliberately more realistic than our Week 2 preview.  
We will do **proper preprocessing** for the IBM Telco dataset, then build classification models and connect their outputs to a business retention decision.

By the end of this seminar, you should be able to:

- clean and prepare the Telco churn data more carefully;
- engineer simple business-informed features;
- create train / validation / test splits;
- build leak-free `Pipeline` and `ColumnTransformer` workflows;
- compare a dummy baseline, Logistic Regression and KNN;
- interpret accuracy, precision, recall, F1, ROC-AUC and PR-AUC;
- experiment with classification thresholds;
- select a decision rule on validation data and evaluate it once on the final test set.

> **Important principle:**  
> A classifier does not directly make a business decision.  
> It estimates risk. The business chooses what action to take.



# 1. Business case

A telecommunications company wants to reduce customer churn.

The Customer Retention Team cannot contact every customer, so the model should help identify customers with high churn risk.

Before modelling, state the decision problem.

| Item | Your answer |
|---|---|
| Stakeholder | |
| Business decision | |
| Unit of observation | |
| Target | |
| Positive class | |
| False positive consequence | |
| False negative consequence | |
| Capacity / constraint | |


In [ ]:

import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

RANDOM_STATE = 42



# 2. Load the IBM Telco Customer Churn dataset

The notebook first looks for a local CSV file. If it is not found, it loads the public IBM sample dataset from GitHub.

Recommended file name if using local data:

`WA_FnUseC_TelcoCustomerChurn.csv`


In [ ]:

LOCAL_FILE = Path("WA_FnUseC_TelcoCustomerChurn.csv")
DATA_URL = (
    "https://raw.githubusercontent.com/IBM/watsonx-ai-samples/"
    "master/cpd4.5/data/customer_churn/WA_FnUseC_TelcoCustomerChurn.csv"
)

if LOCAL_FILE.exists():
    raw = pd.read_csv(LOCAL_FILE)
    print(f"Loaded local file: {LOCAL_FILE}")
else:
    raw = pd.read_csv(DATA_URL)
    print("Loaded IBM Telco Customer Churn from IBM GitHub.")

print(raw.shape)
raw.head()



# 3. Initial data audit

We start with a compact audit. The goal is not to produce every possible chart, but to identify modelling decisions.


In [ ]:

raw.info()


In [ ]:

audit = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "unique_values": raw.nunique(dropna=False),
    "missing_values_pandas": raw.isna().sum()
})

audit


In [ ]:

print("Duplicate rows:", raw.duplicated().sum())
print()
print("Target distribution:")
print(raw["Churn"].value_counts())
print()
print("Target share:")
print(raw["Churn"].value_counts(normalize=True).round(3))



## Student task 1 — Audit interpretation

Answer briefly:

1. Is the target balanced?
2. Why is `customerID` unsuitable as a predictive feature?
3. Which column looks numeric but is stored as text?
4. What is a sensible dummy baseline for this classification problem?

> YOUR ANSWER:



# 4. Proper cleaning for IBM Telco

This dataset looks clean at first glance, but several fields need careful handling.

We will:

1. strip whitespace from text columns;
2. convert `TotalCharges` to numeric;
3. investigate the blank `TotalCharges` rows;
4. set `TotalCharges = 0` for customers with `tenure = 0`;
5. convert `SeniorCitizen` from 0/1 into a categorical Yes/No value;
6. map target `Churn` to 0/1;
7. remove `customerID`.


In [ ]:

df = raw.copy()

# 1. Strip whitespace in text columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

# 2. Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# 3. Investigate missing TotalCharges
missing_total = df[df["TotalCharges"].isna()]
print("Rows with missing TotalCharges:", len(missing_total))
missing_total[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]



## Why not simply fill `TotalCharges` with the median?

In this dataset, the missing `TotalCharges` rows are customers with `tenure = 0`.

That means these are effectively brand-new customers. Since they have not accumulated charges yet, setting `TotalCharges` to **0** is more meaningful than filling with the overall median.

This is an example of **business-aware preprocessing**:

> Understand what the missing value means before choosing an imputation method.


In [ ]:

# 4. Business-aware fix: tenure = 0 implies no accumulated charges yet
mask_total_missing = df["TotalCharges"].isna()

print("Tenure values where TotalCharges is missing:")
print(df.loc[mask_total_missing, "tenure"].value_counts())

df.loc[mask_total_missing & (df["tenure"] == 0), "TotalCharges"] = 0

# If any missing TotalCharges remain, we will leave them for the pipeline median imputer
print("Remaining missing TotalCharges:", df["TotalCharges"].isna().sum())


In [ ]:

# 5. Convert SeniorCitizen into a categorical variable for interpretability
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

# 6. Map target
df["ChurnFlag"] = df["Churn"].map({"No": 0, "Yes": 1})

# 7. Remove original target text later; keep for EDA if needed
df.head()



# 5. Business-informed feature engineering

Feature engineering should be simple, justified, and available at prediction time.

We will create features that are row-wise transformations of information known about the customer:

- `tenure_group`: fixed tenure bands, not quantiles learned from the full data;
- `is_new_customer`: whether tenure is 6 months or below;
- `charges_per_tenure`: approximate average accumulated charge per month of tenure;
- `service_count`: count of subscribed services/add-ons;
- `has_internet`: whether customer has internet service;
- `has_phone`: whether customer has phone service;
- `auto_payment`: whether payment method is automatic.

These are not guaranteed to improve every model, but they make the preprocessing more realistic and interpretable.


In [ ]:

def add_telco_features(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()

    # Fixed business-defined tenure groups
    data["tenure_group"] = pd.cut(
        data["tenure"],
        bins=[-1, 6, 24, 48, 72],
        labels=["0-6 months", "7-24 months", "25-48 months", "49-72 months"]
    ).astype("object")

    data["is_new_customer"] = np.where(data["tenure"] <= 6, "Yes", "No")

    # Avoid division by zero. For tenure=0, TotalCharges is 0 and this ratio is set to 0.
    data["charges_per_tenure"] = np.where(
        data["tenure"] > 0,
        data["TotalCharges"] / data["tenure"],
        0
    )

    internet_addons = [
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
    ]

    # Count services/add-ons marked as Yes
    service_cols = ["PhoneService", "MultipleLines"] + internet_addons
    data["service_count"] = data[service_cols].apply(lambda row: (row == "Yes").sum(), axis=1)

    data["has_internet"] = np.where(data["InternetService"] == "No", "No", "Yes")
    data["has_phone"] = np.where(data["PhoneService"] == "Yes", "Yes", "No")

    data["auto_payment"] = np.where(
        data["PaymentMethod"].isin(["Bank transfer (automatic)", "Credit card (automatic)"]),
        "Yes",
        "No"
    )

    return data

df_fe = add_telco_features(df)

df_fe[[
    "tenure", "tenure_group", "is_new_customer", "MonthlyCharges",
    "TotalCharges", "charges_per_tenure", "service_count",
    "has_internet", "has_phone", "auto_payment", "Churn"
]].head()



## Student task 2 — Feature engineering interpretation

For each engineered feature below, explain why it could plausibly help churn prediction.

1. `tenure_group`
2. `is_new_customer`
3. `charges_per_tenure`
4. `service_count`
5. `auto_payment`

Also state one risk of adding too many engineered features.

> YOUR ANSWER:



# 6. Short EDA connected to modelling

We already studied EDA in Week 2. Here we only inspect signals that directly inform classification and preprocessing.


In [ ]:

# Churn rate by contract type
contract_churn = (
    df_fe.groupby("Contract")["ChurnFlag"]
    .mean()
    .sort_values(ascending=False)
)

contract_churn.plot(kind="barh", title="Churn rate by contract type")
plt.xlabel("Churn rate")
plt.ylabel("Contract")
plt.show()

contract_churn.round(3)


In [ ]:

# Churn rate by tenure group
tenure_group_churn = (
    df_fe.groupby("tenure_group", observed=False)["ChurnFlag"]
    .mean()
)

tenure_group_churn.plot(kind="bar", title="Churn rate by tenure group")
plt.ylabel("Churn rate")
plt.xlabel("Tenure group")
plt.xticks(rotation=35)
plt.show()

tenure_group_churn.round(3)


In [ ]:

# Monthly charges by churn status
df_fe.boxplot(column="MonthlyCharges", by="Churn", grid=False)
plt.title("MonthlyCharges by churn status")
plt.suptitle("")
plt.ylabel("MonthlyCharges")
plt.show()

df_fe.groupby("Churn")["MonthlyCharges"].describe()



## Student task 3 — EDA → modelling implication

Complete the table.

| EDA finding | Evidence | Possible modelling implication |
|---|---|---|
| Churn differs by contract type | | |
| Churn differs by tenure group | | |
| Monthly charges differ by churn status | | |

> YOUR ANSWER:



# 7. Define X and y

We now define the modelling table.

Important exclusions:

- `customerID`: identifier;
- `Churn`: original target text;
- `ChurnFlag`: numerical target, removed from features.

Everything else is candidate input.


In [ ]:

target = "ChurnFlag"

drop_cols = ["customerID", "Churn", "ChurnFlag"]
X = df_fe.drop(columns=drop_cols)
y = df_fe[target]

print("X shape:", X.shape)
print("y mean / churn rate:", round(y.mean(), 3))
X.head()



# 8. Train / validation / test split

Because we will choose thresholds and compare models, we need a validation set.

- **Training set:** fit preprocessing and model parameters.
- **Validation set:** choose threshold/model/hyperparameter.
- **Test set:** final check after decisions are fixed.

The test set should not be used during model selection.


In [ ]:

# First split off the final test set
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

# Then split trainval into training and validation
X_train, X_valid, y_train, y_valid = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.25,  # 0.25 of 80% = 20% of full data
    random_state=RANDOM_STATE,
    stratify=y_trainval
)

print("Train:", X_train.shape, "churn rate:", round(y_train.mean(), 3))
print("Valid:", X_valid.shape, "churn rate:", round(y_valid.mean(), 3))
print("Test: ", X_test.shape,  "churn rate:", round(y_test.mean(), 3))



# 9. Preprocessing columns

For today's models, we will use:

- numerical features: impute median and scale;
- categorical features: impute most frequent and one-hot encode.

For Logistic Regression, scaling is helpful for regularisation and coefficient stability.  
For KNN, scaling is essential because the model uses distances.


In [ ]:

numeric_features = X_train.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

print("Numerical features:", numeric_features)
print()
print("Categorical features:", categorical_features)


In [ ]:

# Compatibility helper for older/newer sklearn versions
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_not_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", ohe)
])

preprocess_scaled = ColumnTransformer(
    transformers=[
        ("num", numeric_scaled, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop"
)

preprocess_not_scaled = ColumnTransformer(
    transformers=[
        ("num", numeric_not_scaled, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop"
)

preprocess_scaled



## Student task 4 — Preprocessing reasoning

Explain why each choice is used:

1. Median imputation for numerical variables.
2. Most-frequent imputation for categorical variables.
3. One-hot encoding for categories.
4. Scaling for Logistic Regression and KNN.
5. `Pipeline` and `ColumnTransformer`.

> YOUR ANSWER:



# 10. Evaluation functions

We will evaluate the **positive class = churn**.

Notice that accuracy alone is not enough.  
For churn, precision and recall are often more informative.


In [ ]:

def evaluate_classifier(name, model, X_data, y_true, threshold=0.50):
    """Evaluate a fitted binary classifier using a chosen probability threshold."""
    proba = model.predict_proba(X_data)[:, 1]
    pred = (proba >= threshold).astype(int)

    return {
        "model": name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, proba),
        "pr_auc": average_precision_score(y_true, proba),
        "flagged_share": pred.mean(),
        "flagged_count": int(pred.sum())
    }

def show_confusion(model, X_data, y_true, threshold=0.50, title="Confusion matrix"):
    proba = model.predict_proba(X_data)[:, 1]
    pred = (proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["No churn", "Churn"])
    disp.plot(values_format="d", cmap="Blues")
    plt.title(title)
    plt.show()
    return cm



# 11. Dummy baseline

A baseline is required.

If our model cannot beat a simple baseline, it is not useful.


In [ ]:

dummy = Pipeline(steps=[
    ("preprocess", preprocess_scaled),
    ("model", DummyClassifier(strategy="most_frequent"))
])

dummy.fit(X_train, y_train)

# DummyClassifier can use predict_proba, so our function works.
baseline_valid = evaluate_classifier("Most-frequent baseline", dummy, X_valid, y_valid, threshold=0.50)
pd.DataFrame([baseline_valid]).round(3)



## Student task 5 — Baseline interpretation

1. What class does the most-frequent baseline predict?
2. Why can its accuracy look acceptable?
3. Why is this baseline not useful for retention?

> YOUR ANSWER:



# 12. Logistic Regression model

We will use `class_weight="balanced"` to make the model pay more attention to the minority churn class.

This is not always automatically the best choice, but it is a useful first model when recall matters.


In [ ]:

log_reg = Pipeline(steps=[
    ("preprocess", preprocess_scaled),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="lbfgs"
    ))
])

log_reg.fit(X_train, y_train)

log_valid = evaluate_classifier("Logistic Regression", log_reg, X_valid, y_valid, threshold=0.50)
pd.DataFrame([baseline_valid, log_valid]).round(3)


In [ ]:

show_confusion(log_reg, X_valid, y_valid, threshold=0.50, title="Logistic Regression on validation set, threshold=0.50")



## Student task 6 — Logistic Regression interpretation

Answer based on the validation results:

1. Did Logistic Regression improve over the baseline?
2. Is accuracy the most important metric here?
3. What do precision and recall tell us in business terms?
4. Does `class_weight="balanced"` guarantee the best business decision?

> YOUR ANSWER:



# 13. Threshold experiment

The model produces churn probabilities. The threshold converts those probabilities into actions.

We choose thresholds on the **validation set**, not the final test set.


In [ ]:

thresholds = [0.70, 0.60, 0.50, 0.40, 0.30, 0.20]

threshold_table = pd.DataFrame([
    evaluate_classifier("Logistic Regression", log_reg, X_valid, y_valid, threshold=t)
    for t in thresholds
])

threshold_table[[
    "threshold", "flagged_count", "flagged_share",
    "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"
]].round(3)



## Student task 7 — Threshold interpretation

Using the threshold table:

1. What happens to recall when the threshold is lowered?
2. What usually happens to precision?
3. Which threshold would you choose if the retention team wants high recall?
4. Which threshold would you choose if the retention team has very limited contact capacity?

> YOUR ANSWER:



# 14. Capacity-based targeting: contact the top 20%

In many retention campaigns, the company does not use a fixed probability threshold.

Instead, it may say:

> We can contact only the top 20% highest-risk customers.

This is often closer to the real business decision.


In [ ]:

def evaluate_top_fraction(name, model, X_data, y_true, fraction=0.20):
    proba = model.predict_proba(X_data)[:, 1]
    cutoff = np.quantile(proba, 1 - fraction)
    pred = (proba >= cutoff).astype(int)

    return {
        "model": name,
        "top_fraction": fraction,
        "cutoff_probability": cutoff,
        "flagged_count": int(pred.sum()),
        "flagged_share": pred.mean(),
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, proba),
        "pr_auc": average_precision_score(y_true, proba),
    }

top20_valid = evaluate_top_fraction("Logistic Regression", log_reg, X_valid, y_valid, fraction=0.20)
pd.DataFrame([top20_valid]).round(3)



## Student task 8 — Capacity-based decision

Assume the retention team can contact only 20% of customers.

1. Why might top-20% targeting be more realistic than using threshold 0.50?
2. How many customers are flagged in the validation set?
3. What do precision and recall mean under this top-20% rule?
4. Would you recommend this decision rule? Why or why not?

> YOUR ANSWER:



# 15. KNN model with proper scaling

KNN uses distances. Scaling is therefore not optional.


In [ ]:

knn_scaled = Pipeline(steps=[
    ("preprocess", preprocess_scaled),
    ("model", KNeighborsClassifier(n_neighbors=15))
])

knn_scaled.fit(X_train, y_train)

knn_valid = evaluate_classifier("KNN scaled, k=15", knn_scaled, X_valid, y_valid, threshold=0.50)

pd.DataFrame([baseline_valid, log_valid, knn_valid]).round(3)



# 16. Demonstrate why scaling matters for KNN

We now fit a KNN model with the same categorical encoding but **without scaling numerical variables**.

This is intentionally included to show how preprocessing affects the model.


In [ ]:

knn_unscaled = Pipeline(steps=[
    ("preprocess", preprocess_not_scaled),
    ("model", KNeighborsClassifier(n_neighbors=15))
])

knn_unscaled.fit(X_train, y_train)

knn_unscaled_valid = evaluate_classifier("KNN unscaled, k=15", knn_unscaled, X_valid, y_valid, threshold=0.50)

pd.DataFrame([knn_unscaled_valid, knn_valid]).round(3)



## Student task 9 — Scaling and KNN

1. Why can unscaled numerical variables distort neighbour selection?
2. Which numerical feature may dominate distance if not scaled?
3. Did scaling improve the KNN validation result here?
4. Would the same scaling issue affect Decision Trees in the same way?

> YOUR ANSWER:



# 17. Choosing k

The value of `k` controls model flexibility.

- very small `k`: flexible, sensitive to noise;
- very large `k`: smoother, but may miss local structure.

We compare several values on the validation set.


In [ ]:

k_values = [1, 3, 5, 11, 15, 25, 51]

knn_results = []

for k in k_values:
    model = Pipeline(steps=[
        ("preprocess", preprocess_scaled),
        ("model", KNeighborsClassifier(n_neighbors=k))
    ])
    model.fit(X_train, y_train)
    knn_results.append(evaluate_classifier(f"KNN k={k}", model, X_valid, y_valid, threshold=0.50))

knn_results_df = pd.DataFrame(knn_results)
knn_results_df[["model", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]].round(3)



## Student task 10 — Choosing k

1. Which value of `k` gives the highest validation F1?
2. Which gives the highest recall?
3. Why should we not choose `k` using the final test set?
4. Does the best `k` depend on the metric?

> YOUR ANSWER:



# 18. Select a decision rule using validation data

For demonstration, we will select:

- model: Logistic Regression;
- decision rule: contact top 20% highest-risk customers.

This is not the only defensible choice.  
The point is that the choice is made using validation data.

Now, and only now, we evaluate once on the final test set.


In [ ]:

final_test_result = evaluate_top_fraction(
    "Logistic Regression, top 20%",
    log_reg,
    X_test,
    y_test,
    fraction=0.20
)

pd.DataFrame([top20_valid, final_test_result]).round(3)


In [ ]:

# Confusion matrix for the final test decision rule
proba_test = log_reg.predict_proba(X_test)[:, 1]
cutoff_test = np.quantile(proba_test, 1 - 0.20)
pred_test_top20 = (proba_test >= cutoff_test).astype(int)

cm = confusion_matrix(y_test, pred_test_top20)
disp = ConfusionMatrixDisplay(cm, display_labels=["No churn", "Churn"])
disp.plot(values_format="d", cmap="Blues")
plt.title("Final test set: Logistic Regression, top 20% highest risk")
plt.show()

print(classification_report(y_test, pred_test_top20, target_names=["No churn", "Churn"]))



## Student task 11 — Final test interpretation

1. Did validation and test performance look similar or very different?
2. How many customers would be contacted in the test set under the top-20% rule?
3. Of those contacted, approximately what share actually churned?
4. Of all churners in the test set, approximately what share were identified?
5. Is this model ready for deployment? What would you investigate next?

> YOUR ANSWER:



# 19. Classification Decision Note

Write a concise decision note for the Customer Retention Director.

Maximum six bullets:

1. business objective and target;
2. baseline performance;
3. best model / decision rule tested;
4. validation evidence;
5. final test evidence;
6. limitation or next step.

> YOUR CLASSIFICATION DECISION NOTE:



# Part B — Independent practice: UCI Bank Marketing

Use the UCI Bank Marketing dataset from Week 2.

**Business question:** Which clients should be prioritised for a future marketing call?  
**Target:** whether the client subscribed to a term deposit.

Repeat the Week 4 workflow:

1. Define the positive class.
2. Remove or justify any variable not known before the call.
3. Build train / validation / test splits.
4. Build a proper preprocessing pipeline.
5. Compare:
   - `DummyClassifier`
   - Logistic Regression
   - KNN with scaling
   - KNN without scaling
6. Report:
   - confusion matrix;
   - accuracy;
   - precision;
   - recall;
   - F1;
   - ROC-AUC;
   - PR-AUC.
7. Run a threshold experiment.
8. Write a six-bullet classification decision note.


In [ ]:

# Optional loader for independent practice
# If ucimlrepo is missing, uncomment the pip install line.

# %pip install -q ucimlrepo

from ucimlrepo import fetch_ucirepo

bank_marketing = fetch_ucirepo(id=222)

bank_X = bank_marketing.data.features.copy()
bank_y = bank_marketing.data.targets.copy()

bank_df = bank_X.copy()
bank_df["target"] = bank_y.iloc[:, 0]

print(bank_df.shape)
bank_df.head()


In [ ]:

# YOUR BANK MARKETING WORK HERE
